# Ultra-Scale Playbook 训练系统 · 第 12/14 课

> 状态：**参考答案版**  
> 一次只完成一课；未通过前不要打开下一课答案。

## 统一完成标准

代码 4 分、Q1～Q3 各 2 分，通过线 8/10。必须解释正确性边界、显存/通信公式中的单位与分片维度；未实际运行的内容只能标记为静态审查。

# 第 12 课：配置搜索与性能诊断

- 对应官方章节：Finding the Best Training Configuration、附录 A1（Profiling）、附录 A3（Compute/Communication Overlap Math）
- 前置：第 11 课
- 状态：未开始

## 本课目标

完成后你需要能够：

- 复述教材的三步配置流程，并说明每步的检查项。
- 计算 MFU 与 HFU，解释两者的区别与适用场景。
- 使用附录 A3 的重叠条件（t_comm/t_compute ≤ 1）判断 DP/ZeRO-3/TP/PP 的通信能否被隐藏。
- 用 PyTorch profiler / torch.cuda.Event / ncu 定位"装得下但慢"的瓶颈。

## 核心概念

### 1. 三步配置流程（教材）

1. **Step 1：让一个训练步装进内存**。GPU 富裕：<10B 单技术（TP 或 ZeRO-3/DP+全量重计算）；10–100B：TP=8+PP、TP=8+ZeRO-3 或纯 ZeRO-3；512+ 卡纯 DP/ZeRO-3 低效 → 加 TP/PP；1024+ 卡建议 TP=8+ZeRO-2+PP。GPU 紧张：全量重计算、梯度累积。超长序列加 CP；MoE 加 EP。
2. **Step 2：达到目标 gbs**。`gbs_tokens = mbs × grad_acc × dp × seq`。不够 → 加 dp / grad_acc（长序列可加 CP）；太多 → 减 dp。
3. **Step 3：优化吞吐**。TP 升到节点大小；ZeRO-3 加 dp；DP 通信瓶颈 → 转 PP；逐维度尝试；调 mbs。

### 2. MFU 与 HFU


In [ ]:
前向+反向必需 FLOPs ≈ 6 × num_tokens × num_params
MFU = 6·tokens·params / (step_time × peak_flops)
HFU = 6·tokens·params × (1 + recompute_ratio) / (step_time × peak_flops)


- **MFU（model FLOPs utilization）**：只算模型必需运算，衡量"实现/硬件对模型的效率"——重计算不计入。比较 GPU 时用它（重计算少但更快的机器应被奖励）。
- **HFU（hardware FLOPs utilization）**：算上重计算等真实发生的运算，衡量"硬件真实利用率"。
- 陷阱：HFU 高不一定训练快（可能在做大量重计算）；MFU 更能反映训练时间。

### 3. 重叠数学（附录 A3，面试可手推）

通信能完全隐藏的条件：`t_comm / t_compute ≤ 1`。

| 并行 | 每步通信 | t_comm/t_compute |
|---|---|---|
| DP（ZeRO-0） | 梯度 ≈ 参数量 | `num_params/(2·num_tokens) · (DP−1)/DP · F/B` |
| ZeRO-3 | 3×参数量/DP（逐层） | `1/(2·seq·mbs) · (DP−1)/DP · F/B` |
| TP | 每层 8·seq·mbs·h/TP | `(TP−1)/(2h) · F/B` |
| PP | 每微批次 4·seq·mbs·h | `F/(32·h·n_layers_next · B)` |

其中 F = peak_flops，B = peak_bw。观察：

- DP/ZeRO-3 的比值随 tokens 增多而减小（大批次藏得住）；
- TP/PP 的比值与 batch/seq **无关**，只取决于 h 与硬件算力/带宽比——这解释了为什么 TP 跨节点必炸。

### 4. 诊断工具与常见瓶颈

- **torch.profiler**：CPU/CUDA 活动时间线；看通信是否与计算重叠、GPU 是否空闲、kernel launch 开销。
- **torch.cuda.Event + synchronize**：CUDA 异步，python time 测不出真时长。
- **ncu（Nsight Compute）**：单 kernel 级的吞吐/占用率/访存分析。
- **torch.cuda.max_memory_allocated / reserved**：峰值显存与缓存分配器预留之差（碎片）。
- 教材经验：benchmark 上万配置时，Slurm 强杀、进程清理失败、NCCL debug 日志、CUDA allocator 行为是最耗时的工程问题。

## 具体演示

H100（peak ≈ 989 TFLOPS BF16，NVLink ≈ 900 GB/s），7B 模型，全局 gbs=4M tokens，DP=128（每卡每步 31250 token）：


In [ ]:
t_comm(环 all-reduce) = 2×14GB×127/128 / 900GB/s ≈ 31 ms
t_compute(本卡 backward) = 4×31250×7e9 / 989e12 ≈ 0.88 s
比值 ≈ 0.035 → 通信完全藏得住


若每卡 token 太少（如 mbs·seq=512），t_compute 掉到 ~15 ms，比值 >2 → 通信暴露。教材 benchmark 固定 gbs=1M tokens 去扩 dp，每卡 mbs·seq 被迫变小，正是这个机制导致"节点越多效率下降"。而 ZeRO-3 的层级 prefetch 在 seq 长、mbs 大时比值很小，可以藏住。

## 代码填空题

实现 MFU/HFU 与重叠条件判断。


In [ ]:
def step_flops(num_params: int, num_tokens: int) -> float:
    """
    前向+反向必需 FLOPs ≈ 6·num_tokens·num_params
    （前向 2、反向 4；忽略 attention 的 seq² 二次项，教材 A2 注释说明）。
    """
    return ______


def mfu_hfu(
    num_params: int, num_tokens: int, step_time_s: float,
    peak_flops: float, recompute_extra: float = 0.0,
) -> tuple[float, float]:
    """
    recompute_extra：重计算带来的额外前向比例（full 重计算约 0.3–0.4）。
    MFU 不含重计算；HFU 含。
    """
    required = step_flops(num_params, num_tokens)
    mfu = required / (step_time_s * peak_flops)
    hfu = ______                      # 填空：把重计算计入真实 FLOPs
    return mfu, hfu


def dp_overlap_ratio(
    num_params: int, num_tokens: int, dp: int,
    peak_flops: float, peak_bw: float,
    bytes_per_param: int = 2,
) -> float:
    """
    DP 梯度 all-reduce 能否被本卡 backward 隐藏：直接用时间比 t_comm / t_compute。

      t_comm    = 2·num_params·bytes_per_param·(DP-1)/DP / peak_bw
        （ring all-reduce 的每卡发送量 ≈ 2×梯度字节；DP 很大时 (DP-1)/DP ≈ 1）
      t_compute = 4·num_tokens·num_params / peak_flops
        （本卡负责的那部分 backward；注意 num_tokens 是【每卡每步】token 数
          = mbs·seq，不是全局 gbs —— 这是最容易算错的地方）

    比 < 1 说明通信可以被完全重叠。教材 A3 的紧凑形式
    num_params/(2·num_tokens)·(DP-1)/DP·F/B 是它的等价变形，
    但换算时务必保持单位一致（bytes vs 元素数、全局 vs 每卡）。
    """
    t_comm = ______                # 填空：ring all-reduce 每卡发送量/带宽
    t_comp = ______                # 填空：本卡 backward FLOPs/峰值算力
    return t_comm / t_comp


def zero3_overlap_ratio(
    seq_len: int, mbs: int, dp: int,
    peak_flops: float, peak_bw: float,
    layer_params: int = 16 * 4096 * 4096,
) -> float:
    """
    ZeRO-3 逐层 prefetch 能否藏住一层的参数 all-gather：

      t_comm    = 2·layer_params·bytes_per_param·(DP-1)/DP / peak_bw
      t_compute = 2·seq·mbs·layer_params / peak_flops   （该层前向计算）

    比 < 1 说明第 n+1 层的参数 all-gather 可以藏在第 n 层的计算后面。
    """
    t_comm = ______                # 填空
    t_comp = ______                # 填空
    return t_comm / t_comp


if __name__ == "__main__":
    H100_F, H100_B = 989e12, 900e9   # BF16 TFLOPS、NVLink GB/s

    # 7B、全局 gbs=4M tokens、dp=128 → 每卡每步 tokens = 4e6/128 = 31250
    per_gpu = 4_000_000 // 128
    mfu, hfu = mfu_hfu(7e9, per_gpu, 2.0, H100_F, recompute_extra=0.35)
    print(f"MFU = {mfu:.3f}, HFU(含35%重计算) = {hfu:.3f}")

    for dp in (32, 128, 512):
        r = dp_overlap_ratio(7e9, 4_000_000 // dp, dp, H100_F, H100_B)
        print(f"DP={dp:4d}  每卡tokens={4_000_000//dp:6d}  t_comm/t_compute = {r:.2f}  "
              f"{'可隐藏' if r <= 1 else '不可隐藏'}")

    # 小 mbs·seq 时通信藏不住（每卡计算太少）
    r = dp_overlap_ratio(7e9, 512, 128, H100_F, H100_B)
    print(f"DP=128 但每卡 tokens=512: {r:.2f}  {'可隐藏' if r <= 1 else '不可隐藏'}")

    # ZeRO-3：seq=4096、mbs=2，一层参数 16h²（h=4096）
    r = zero3_overlap_ratio(4096, 2, 128, H100_F, H100_B)
    print(f"ZeRO-3(seq=4096,mbs=2,dp=128): {r:.3f}  {'可隐藏' if r <= 1 else '不可隐藏'}")


## 三个问答题


### Q1

MFU 与 HFU 的分子差在哪里？教材为什么说"重计算会抬高 HFU 但不一定代表训练更快"？如果要比较两台硬件谁的训练实现更高效，应该看哪个指标？为什么？


### Q2

某配置"装得下但训练慢"。请给出你的诊断流程：至少包含 1) 用什么工具看什么；2) 两个最可能的瓶颈假设及各自的证据特征（例如 trace 中通信与计算是否重叠、kernel 是否过小）。测量显存时 `max_memory_allocated` 与 `max_memory_reserved` 的差说明什么？


### Q3

用附录 A3 的公式回答：为什么"目标 gbs 固定时把 DP 加得很大"会导致 DP 通信藏不住（提示：每卡通信量 ≈ 2×模型梯度、与 dp 无关，而每卡 backward 时间正比于 mbs·seq）？为什么 TP 的重叠条件与 batch、seq 无关？这两个结论分别指导了什么工程决策（第 4/6/11 课的规则）？

## 检查与通过标准

总分 10 分：代码正确 4 分（FLOPs 公式、HFU、两个重叠比例）、三题各 2 分、通过线 8 分。

一票否决项：

- MFU/HFU 分子混淆（重计算计入谁）。
- 把 t_comm/t_compute 的方向搞反（>1 说成可隐藏）。
- 认为 python time 能准确测量 CUDA kernel 耗时。
- 只会报"慢"说不出定位方法。


## 参考答案（仅 answer 分支）

先完成题目再核对；核对后必须解释关键公式，并改一个规模重新计算。

In [ ]:
def step_flops(num_params: int, num_tokens: int) -> float:
    """
    前向+反向必需 FLOPs ≈ 6·num_tokens·num_params
    （前向 2、反向 4；忽略 attention 的 seq² 二次项，教材 A2 注释说明）。
    """
    return 6 * num_tokens * num_params


def mfu_hfu(
    num_params: int, num_tokens: int, step_time_s: float,
    peak_flops: float, recompute_extra: float = 0.0,
) -> tuple[float, float]:
    """
    recompute_extra：重计算带来的额外前向比例（full 重计算约 0.3–0.4）。
    MFU 不含重计算；HFU 含。
    """
    required = step_flops(num_params, num_tokens)
    mfu = required / (step_time_s * peak_flops)
    hfu = required * (1 + recompute_extra) / (step_time_s * peak_flops)                      # 填空：把重计算计入真实 FLOPs
    return mfu, hfu


def dp_overlap_ratio(
    num_params: int, num_tokens: int, dp: int,
    peak_flops: float, peak_bw: float,
    bytes_per_param: int = 2,
) -> float:
    """
    DP 梯度 all-reduce 能否被本卡 backward 隐藏：直接用时间比 t_comm / t_compute。

      t_comm    = 2·num_params·bytes_per_param·(DP-1)/DP / peak_bw
        （ring all-reduce 的每卡发送量 ≈ 2×梯度字节；DP 很大时 (DP-1)/DP ≈ 1）
      t_compute = 4·num_tokens·num_params / peak_flops
        （本卡负责的那部分 backward；注意 num_tokens 是【每卡每步】token 数
          = mbs·seq，不是全局 gbs —— 这是最容易算错的地方）

    比 < 1 说明通信可以被完全重叠。教材 A3 的紧凑形式
    num_params/(2·num_tokens)·(DP-1)/DP·F/B 是它的等价变形，
    但换算时务必保持单位一致（bytes vs 元素数、全局 vs 每卡）。
    """
    t_comm = 2 * num_params * bytes_per_param * (dp - 1) / dp / peak_bw                # 填空：ring all-reduce 每卡发送量/带宽
    t_comp = 4 * num_tokens * num_params / peak_flops                # 填空：本卡 backward FLOPs/峰值算力
    return t_comm / t_comp


def zero3_overlap_ratio(
    seq_len: int, mbs: int, dp: int,
    peak_flops: float, peak_bw: float,
    layer_params: int = 16 * 4096 * 4096,
) -> float:
    """
    ZeRO-3 逐层 prefetch 能否藏住一层的参数 all-gather：

      t_comm    = 2·layer_params·bytes_per_param·(DP-1)/DP / peak_bw
      t_compute = 2·seq·mbs·layer_params / peak_flops   （该层前向计算）

    比 < 1 说明第 n+1 层的参数 all-gather 可以藏在第 n 层的计算后面。
    """
    t_comm = 2 * layer_params * 2 * (dp - 1) / dp / peak_bw                # 填空
    t_comp = 2 * seq_len * mbs * layer_params / peak_flops                # 填空
    return t_comm / t_comp


if __name__ == "__main__":
    H100_F, H100_B = 989e12, 900e9   # BF16 TFLOPS、NVLink GB/s

    # 7B、全局 gbs=4M tokens、dp=128 → 每卡每步 tokens = 4e6/128 = 31250
    per_gpu = 4_000_000 // 128
    mfu, hfu = mfu_hfu(7e9, per_gpu, 2.0, H100_F, recompute_extra=0.35)
    print(f"MFU = {mfu:.3f}, HFU(含35%重计算) = {hfu:.3f}")

    for dp in (32, 128, 512):
        r = dp_overlap_ratio(7e9, 4_000_000 // dp, dp, H100_F, H100_B)
        print(f"DP={dp:4d}  每卡tokens={4_000_000//dp:6d}  t_comm/t_compute = {r:.2f}  "
              f"{'可隐藏' if r <= 1 else '不可隐藏'}")

    # 小 mbs·seq 时通信藏不住（每卡计算太少）
    r = dp_overlap_ratio(7e9, 512, 128, H100_F, H100_B)
    print(f"DP=128 但每卡 tokens=512: {r:.2f}  {'可隐藏' if r <= 1 else '不可隐藏'}")

    # ZeRO-3：seq=4096、mbs=2，一层参数 16h²（h=4096）
    r = zero3_overlap_ratio(4096, 2, 128, H100_F, H100_B)
    print(f"ZeRO-3(seq=4096,mbs=2,dp=128): {r:.3f}  {'可隐藏' if r <= 1 else '不可隐藏'}")


# 第 12 课参考答案（复盘用，先提交再打开）

## 补全后的代码（关键填空处）


```python
def step_flops(num_params, num_tokens):
    return 6 * num_tokens * num_params          # 前向 2 + 反向 4

def mfu_hfu(num_params, num_tokens, step_time_s, peak_flops, recompute_extra=0.0):
    required = step_flops(num_params, num_tokens)
    mfu = required / (step_time_s * peak_flops)
    hfu = required * (1 + recompute_extra) / (step_time_s * peak_flops)
    return mfu, hfu

def dp_overlap_ratio(num_params, num_tokens, dp, peak_flops, peak_bw, bytes_per_param=2):
    # 每卡每步：comm ≈ 2×梯度字节（ring all-reduce），compute = 本卡 backward（mbs·seq tokens）
    t_comm = 2 * num_params * bytes_per_param * (dp - 1) / dp / peak_bw
    t_comp = 4 * num_tokens * num_params / peak_flops
    return t_comm / t_comp

def zero3_overlap_ratio(seq_len, mbs, dp, peak_flops, peak_bw, layer_params=16*4096*4096):
    t_comm = 2 * layer_params * 2 * (dp - 1) / dp / peak_bw   # 一层参数的 all-gather
    t_comp = 2 * seq_len * mbs * layer_params / peak_flops    # 该层前向
    return t_comm / t_comp
```


运行结果（H100：989 TFLOPS / 900 GB/s，7B，gbs=4M，grad_acc=1）：


```python
DP=32  每卡tokens=125000  t_comm/t_compute = 0.009  可隐藏
DP=128 每卡tokens= 31250  t_comm/t_compute = 0.035  可隐藏
DP=512 每卡tokens=  7812  t_comm/t_compute = 0.139  可隐藏
DP=128 但每卡tokens=512: 2.13                     不可隐藏
ZeRO-3(seq=4096,mbs=2,dp=128): 0.267              可隐藏
```


MFU/HFU 示例（每卡 31250 tokens、2.0 s/step）：MFU ≈ 0.66，HFU（+35% 重计算）≈ 0.90。

## 三个问答题要点


### Q1

- MFU 分子 = 模型必需 FLOPs（6·tokens·params，不含重计算）；HFU 分子 = 硬件真实执行 FLOPs（含重计算）→ 同一硬件上 HFU ≥ MFU；
- 重计算抬高 HFU 但训练可能更慢：HFU 衡量"硬件被用满的程度"而非"端到端快慢"；比较实现/硬件效率用 MFU（教材："更少重计算但更快的机器应被奖励"）；
- 补充：MFU 是模型视角（与实现无关），HFU 是硬件视角（包含实现引入的额外运算）。


### Q2

诊断流程（要点）：

1. 显存：`max_memory_allocated` vs `max_memory_reserved`——差是 caching allocator 预留/碎片（大则调 `PYTORCH_CUDA_ALLOC_CONF`、减 batch、或查长驻 buffer）；
2. 时间线：torch.profiler（CPU+CUDA）看通信是否与计算重叠、GPU 空闲段（等待通信/数据加载）、kernel 规模与启动开销（过小过碎 → torch.compile/融合）；
3. 单 kernel：ncu 看吞吐/占用率/访存模式；
4. 先算后测：用 t_comm/t_compute 预判该配置是否处在可隐藏区间，避免盲目调参。


### Q3

- 机制：每卡通信量 ≈ 2×模型梯度字节，与 dp 无关；每卡 backward 时间 ∝ 4·(mbs·seq)·Ψ/F。gbs 固定时加 dp → grad_acc 或 mbs 被迫减小 → 每卡 backward 变短而通信不变 → 比值随 dp 增大、超过 1（教材 benchmark 固定 gbs=1M 正是这个机制）；
- TP：重叠条件 (TP−1)/(2h)·F/B 不含 seq/mbs——因为 TP 通信发生在每层内部，与单层计算竞争：单层计算 ∝ seq·mbs·h²、通信 ∝ seq·mbs·h，h²/h 抵消了 batch 项；
- 工程决策：a) 大批次/长序列优先 DP/ZeRO-3（通信可隐藏），小 mbs 场景慎用高 DP；b) TP 只用于节点内切模型/激活，其效率由硬件算力带宽比决定，跨节点必炸（第 6/11 课规则）。

## 通过要点

- 重叠条件：t_comm/t_compute ≤ 1 才可完全隐藏。
- 单位陷阱：tokens 用每卡每步的 mbs·seq；字节与元素数要一致；教材 A3 紧凑公式是等价变形，面试手推时直接用秒为单位最稳。
- 测量：CUDA 异步 → torch.cuda.Event/synchronize 或 profiler，不能用 python time。
- MFU 不含重计算，HFU 含。


## 官方主参考

- [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
- [PyTorch distributed documentation](https://pytorch.org/docs/stable/distributed.html)